# Gesture SVM Training

Trains an SVM on self-collected HOG features from hand ROI images.
Collect data first with collect_gesture_data.py, then run this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, json
sys.path.append('..')
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print('Ready')

In [ ]:
# Load data
csv_path = '../data/gesture/raw/landmarks.csv'
df = pd.read_csv(csv_path)
print(f'Samples: {len(df)}')
print(f'Classes:\n{df["gesture"].value_counts()}')

In [ ]:
# Preprocess
gesture_map = {g: i for i, g in enumerate(df['gesture'].unique())}
X = df.iloc[:, 1:].values
y = df['gesture'].map(gesture_map).values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f'Train: {len(X_train)}, Val: {len(X_val)}')

In [ ]:
# Grid search
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1],
    'kernel': ['rbf'],
}
grid = GridSearchCV(SVC(probability=True, random_state=42),
                     param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')

In [ ]:
# Evaluate
best = grid.best_estimator_
y_pred = best.predict(X_val_scaled)
acc = np.mean(y_pred == y_val)
print(f'Validation accuracy: {acc:.4f}')
print()
print(classification_report(y_val, y_pred, target_names=list(gesture_map.keys())))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(gesture_map.keys()),
            yticklabels=list(gesture_map.keys()))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Gesture Confusion Matrix')
plt.show()

In [ ]:
# Save
joblib.dump(best, '../models/gesture_svm.pkl')
joblib.dump(scaler, '../models/gesture_scaler.pkl')
print('Saved to models/')